### The following cells are modified from Ilya's Code to fit into our project

Category: Appliances

In [ ]:
import os
import duckdb
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [ ]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
CATEGORY = "Appliances"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = RAW_DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = RAW_DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = RAW_DATA_DIR / f"{CATEGORY}_merged.parquet"

In [ ]:
print(os.listdir())
print(RAW_DATA_DIR)
print(REVIEWS_URL)

### Initialize an in-memory DB connection

In [ ]:
c2 = duckdb.connect()

Review data:
- keep `rating, title, text, parent_asin, helpful_vote` columns only while downloading data
- other columns seems useless

In [ ]:
c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()

Meta data
- Drop images and videos while downloading data since we can't process it through links.
- other columns seems useful

In [ ]:
c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()

### download data in parquet if it doesn't exist in the directory

In [ ]:
review_data_file = f'{CATEGORY}_reviews_raw.parquet'
meta_data_file = f'{CATEGORY}_meta_raw.parquet'

if review_data_file not in os.listdir(RAW_DATA_DIR):
    print("Downloading Review Data\nCategory: {}".format(CATEGORY))
    c2.execute(f"""
          COPY (SELECT rating, title, text, parent_asin, helpful_vote FROM read_json_auto('{REVIEWS_URL}'))
          TO '{RAW_DATA_DIR}/{review_data_file}'
          (FORMAT PARQUET, COMPRESSION ZSTD)
      """)
    print("Done")
else:
    print("Review data for {} already downloaded".format(CATEGORY))

In [ ]:
c2.execute(f"SELECT * FROM read_parquet('{RAW_DATA_DIR}/{review_data_file}') LIMIT 5").df()

In [ ]:
if meta_data_file not in os.listdir(RAW_DATA_DIR):
    print("Downloading Meta Data\nCategory: {}".format(CATEGORY))
    c2.execute(f"""
    COPY (SELECT * EXCLUDE (images, videos) FROM read_json_auto('{META_URL}', union_by_name=true))
    TO '{RAW_DATA_DIR}/{meta_data_file}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print("Done")
else:
    print("Meta data for {} already downloaded".format(CATEGORY))


In [ ]:
c2.execute(f"SELECT * FROM read_parquet('{RAW_DATA_DIR}/{meta_data_file}') LIMIT 5").df()

### Merging the two files by joining on `parent_asin`

In [ ]:
merged_data_file = f'{CATEGORY}_merged.parquet'

if merged_data_file not in os.listdir(PROCESSED_DATA_DIR):
    print("Merging Review/Meta Data\nCategory: {}".format(CATEGORY))
    c2.execute(f"""
        COPY (
            SELECT r.*, m.title AS product_title, m.price,
                        m.average_rating, m.main_category, m.store
            FROM read_parquet('{RAW_DATA_DIR}/{review_data_file}') r
            LEFT JOIN read_parquet('{RAW_DATA_DIR}/{meta_data_file}') m USING (parent_asin)
        )
        TO '{PROCESSED_DATA_DIR}/{merged_data_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print("Done")
else:
    print("Merged data for {} is ready".format(CATEGORY))

In [ ]:
merged_data = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')").df()
merged_data

### EDA

#### Unique Product Count

In [ ]:
c2.execute(f"""
    SELECT 
        COUNT(DISTINCT parent_asin) AS 'Number of Unique Products' 
    FROM 
        read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
""").df()

#### Number of reviews for each product

In [ ]:
review_counts = c2.execute(f"""
    SELECT 
        parent_asin,
        COUNT(*) AS review_count
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
    GROUP BY parent_asin
    ORDER BY review_count DESC
""").df()
review_counts

#### Distribution of Review Counts

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
# sns.histplot(df_counts["review_count"], bins=50, log_scale=True)
plt.hist(review_counts["review_count"])
plt.title("Distribution of Reviews per Product")
plt.xlabel("Review Count")
plt.yscale('log')
plt.ylabel("Frequency (log scale)")
plt.show()

#### Distribution of Length of Reviews

In [ ]:
text_length = c2.execute(f"""
    SELECT LENGTH(text) AS text_len
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
    WHERE text IS NOT NULL
    ORDER BY text_len
""").df()

plt.figure(figsize=(8, 5))
plt.hist(text_length["text_len"])
plt.title("Distribution of Review Text Length")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.yscale("log")
plt.show()
text_length

In [ ]:
total_text_length = c2.execute(f"""
    SELECT 
        parent_asin,
        SUM(LENGTH(text)) AS total_text_size
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
    GROUP BY parent_asin
    ORDER BY total_text_size DESC
""").df()

plt.figure(figsize=(8, 5))
plt.hist(total_text_length["total_text_size"])
plt.title("Distribution of Review Text Length")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.yscale("log")
plt.show()
total_text_length

#### Duplicate Reviews

In [ ]:
c2.execute(f"""
    SELECT 
        text,
        COUNT(*) as cnt
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
    GROUP BY text
    HAVING cnt > 1
    ORDER BY cnt DESC
""").df()

#### Missing Values

In [ ]:
c2.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE product_title IS NULL) AS missing_product_title,
        COUNT(*) FILTER (WHERE text IS NULL) AS missing_text,
        COUNT(*) FILTER (WHERE main_category IS NULL) AS missing_category,
        COUNT(*) FILTER (WHERE store IS NULL) AS missing_store,
        COUNT(*) FILTER (WHERE price IS NULL) AS missing_price
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
""").df()

### Build Document for each product

In [ ]:
print(merged_data.columns)

#### Group reviews by products

In [ ]:
products = c2.execute(f"""
    SELECT 
        parent_asin,
        ANY_VALUE(product_title) AS product_title,
        ANY_VALUE(main_category) AS main_category,
        ANY_VALUE(store) AS store,
        ANY_VALUE(price) AS price,
        ANY_VALUE(average_rating) AS avg_rating,
        LIST(text) AS reviews,
        LIST(title) AS review_titles,
        LIST(helpful_vote) AS helpful_votes
    FROM read_parquet('{PROCESSED_DATA_DIR}/{merged_data_file}')
    GROUP BY parent_asin;
""").df()
products

#### Preprocessing Reviews then add to document
- ignore short meaning less reviews
- truncate reviews by number of characters
- only include top reviews ranked by `helpful_vote`
- remove duplicated reviews

In [ ]:
def build_document(product, max_reviews=20, min_len=30, max_chars=None):
    reviews = product['reviews']
    review_titles = product['review_titles']
    helpful_votes = product['helpful_votes']

    # Combine and sort reviews by helpful votes in descending order
    combined = list(zip(helpful_votes, review_titles, reviews))
    combined = sorted(combined, key=lambda x: x[0] if x[0] is not None else 0, reverse=True)

    review_lines = []
    logged_review = []
    for vote, title, review in combined:
        # filter short reviews
        if not review or len(review) < min_len:
            continue
        
        review = review[:max_chars]  # truncate long reviews
        
        # remove duplicate reviews
        if review not in logged_review:
            review_lines.append(f"- ({vote} votes) {title}: {review}")
            logged_review.append(review)
        
        # limit number of reviews
        if len(review_lines) >= max_reviews:
            break

    review_block = "\n".join(review_lines)

    doc = f"""
Title: {product.get('product_title', '')} {product.get('product_title', '')}
Category: {product.get('main_category', '')}
Store: {product.get('store', '')}
Price: {product.get('price', '')}
Average Rating: {product.get('avg_rating', '')}

Reviews:
{review_block}
""".strip()

    return doc


In [ ]:
print(build_document(products.loc[0]))

In [ ]:
print(build_document(products.loc[10]))

#### Convert all products into documents

In [ ]:
import pickle

document_id_file = f"{CATEGORY}_doc_ids.pkl"
documents_file = f"{CATEGORY}_product_documents.pkl"

if document_id_file not in os.listdir(PROCESSED_DATA_DIR):
    print("Downloading document ids\nCategory: {}".format(CATEGORY))
    
    doc_ids = products["parent_asin"].tolist()

    with open(f"{PROCESSED_DATA_DIR}/{document_id_file}", "wb") as f:
        pickle.dump(doc_ids, f)

    print("Done")
else:
    print("Document id for {} is ready".format(CATEGORY))

if documents_file not in os.listdir(PROCESSED_DATA_DIR):
    print("Downloading documents\nCategory: {}".format(CATEGORY))
    
    documents = list(map(build_document, products.to_dict("records")))

    with open(f"{PROCESSED_DATA_DIR}/{documents_file}", "wb") as f:
        pickle.dump(documents, f)

    print("Done")
else:
    print("Document id for {} is ready".format(CATEGORY))

In [ ]:
# corpus = dict(zip(doc_ids, documents))